# Docker Compose

In this notebook, we will learn how to use `docker compose` to work with **multi-container** applications.

Docker Compose:

- Is a tool included with Docker.
- Is invoked with `docker compose <COMMAND>`, where `<COMMAND>` is a Docker Compose command.
- Enables you to define resources (**services**, **volumes** and **networks**) in a YAML file.
  - A **service** is just a **container** based on an **image** and is defined under the `services` section in a Docker Compose YAML file.
  - A **volume** is defined under the `volumes` section in a Docker Compose YAML file, and is referenced from a **service** under the `services` section.
  - A **network** is defined under the `networks` section in a Docker Compose YAML file, and is referenced from a **service** under the `services` section.
  - A Docker Compose YAML file is usually called `compose.yaml` (or `compose.yml`) or `docker-compose-yaml` (or `docker-compose-yml`).
    - Although, any naming convention is allowed.
  
  <img src="notebook_images/basic_docker_compose_file.png" width="600" />

- Let's you create and start all **resources** in a YAML file with one command `docker compose up`.
- Let's you stop and remove all **resources** in a YAML file with one command `docker compose down`.

---

## Examine a Docker Compose File

Let's examine a Docker Compose file for the `Flixtube.Metadata` microservice.

- Expand the folder `flixtube -> Flixtube.Metadata`.
- Right-click the file `docker-compose.yml` and choose `Open to the side`.

**YAML File Structure**

- At the top of the Docker Compose YAML file, we see the start of the `services` section.
- Indented (2 blank spaces) under the `services` section, 3 services `rabbit`, `db` and `metadata` are defied.
  - Their definitions are idented under each of the services.
- At the bottom of the YAML file, we see the `volumes` section.
  - Indented (2 blank spaces) under the `volumes` section, 2 named volumes `rabbit_data` and `sqlserver_data` are defied.
    - Their definitions are idented under each of the volumes.
- We could also add a `networks` section.
  - Indented (2 blank spaces) under the `networks` section, we could define one or more networks, e.g. `flixtube:`
  - Their definitions would be idented under each of the networks, e.g. `driver: bridge`

**The `rabbit` service**

- Indented under the `services` section, the `rabbit` service is defined, where `rabbit` is the **name** of the service.
- Indented under the `rabbit` service, the service is configured.
  - `image` specifies the Docker image to use for the service.
    - In this case the image is `rabbitmq:4.0.2-management`.
  - `container_name` specifies the name to use for the container based on the image.
  - In this case the container name is `rabbit`.
  - `hostname` specifies a host name for the container.
    - In this case the hostname is `rabbithost`.
  - `ports` specifies a list of port mappings in the format `HOST_PORT:SERVICE_PORT`.
    - In this case there are two port mappings `5672:5672` and `15672:15672`.
    - This has the same functionality as the `-p` flag in the `docker run` command.
  - `environment` specifies a list of environment variables in the format `KEY:VALUE`.
    - In this case four environment variables are defined, e.g. `RABBITMQ_DEFAULT_USER=rabbit_user` where `RABBITMQ_DEFAULT_USER` is the environment variable's `KEY` (name) and `rabbit_user` is its value.
    - This has the same functionality as the `-e` flag in the `docker run` command.
  - `volumes` specifies a list of volue mappings in the format `HOST_FOLDER_PATH:SERVICE_FOLDER_PATH` (a bind mount) or `NAMED_VOLUME:SERVICE_FOLDER_PATH` (a named volume).
    - In this case one named volume mapping `rabbit_data:/var/lib/rabbitmq/mnesia/` is defined, where `rabbit_data` refers to the named volume defined under the `volumes` section, and `/var/lib/rabbitmq/mnesia/` is the folder path in the service (i.e. in the container).
    - This has the same functionality as the `-v` flag in the `docker run` command.
  - `restart` specifies if Docker should automatically restart the service (container) if it stops.
    - In this case its value `always` means Docker should always restart the service (container) if it stops.

**Note!**
- A Docker Compose will automatically create a DNS Name for a service's:
  - **name**
  - `container_name`
  - `hostname`
- For the `rabbit` service, this means any other service on the same network as the `rabbit` service, can communicate with the `rabbit` service using its IPAddress, or any of the DNS Names, i.e. the service **name** `rabbit`, the `container_name` `rabbit` or the `hostname` `rabbithost`, e.g. `rabbit:5672` (with the port included).

**The `db` service**

- Indented under the `services` section, the `db` service is defined, where `db` is the **name** of the service.
- Indented under the `db` service, the service is defined in a simlar way as the `rabbit` service.
- Notice the `db` service specifies `sqlserver` as its `container_name` (so the service name and container name don't have to be the same, but a good practice is to keep them the same).
- The `db` service also sets SQLServer-speficic ports and environment variables, and also sets `restart` to `always`.
- Also notice the named volume mapping `sqlserver_data:/var/opt/mssql`, where `sqlserver_data` refers to the named volume defined under the `volumes` section, and `/var/opt/mssql` is the folder path in the service (i.e. in the container).

**The `metadata` service**

- Indented under the `services` section, the `metadata` service is defined, where `metadata` is the **name** of the service.
- Indented under the `metadata` service, the service is defined in a simlar way as the `rabbit` and `db` services but with some noticable differences:
  - `image` specifies `metadata` as the name of the image (which is the same as `metadata:latest`).
  - `build` defines a section for building an image.
    - `context` specifies the path to the folder containing the necessary files to build the image.
    - `dockerfile` specifies a path (relative to `context`) to a Dockerfile used to build the image.
  - The current settings will build an image using the Dockerfile `Dockerfile-dev` in the current folder `./`, which will result in an image called `metadata:latest`.
    - The `image` setting is set to `metadata` (which is the same as `metadata:latest`), so the service will be based on the `metadata:latest` image.
  - `depends_on` is a list of **service names** on which the current service depends.
    - In this case the list includes the service names `db` and `rabbit`.
    - This means that Docker Compose won't try to start the `metadata` service before the `db` and `rabbit` services have started.
  - `volumes` defines a **bind mount** mapping (i.e. NOT a named volume mapping as with the `db` and `rabbit` services).
    - In this case `./:/src` means the current folder `./` on the host is mapped to the `/src` folder in the service (container).
    - This has the same functionality as the `-v` flag in the `docker run` command when a bind mount mapping is used.

**The `volumes` section**

- Indented at the  **same** level as the `services` section, the `volumes` section is defined.
  - Indented under the `volumes` section, the two named volumes `rabbit_data` and `sqlserver_data` are defined.
    - The `rabbit` service references (uses) the named volume `rabbit_data`.
    - The `db` service references (uses) the named volume `sqlserver_data`. 

---

## Using Docker Compose

The Docker Compose tool is part of Docker, and is invoked as a subcommand to `docker`, i.e. as `docker compose`.

To show the Docker Compose help, we can use the command `docker compose --help`, which displays common Docker Compose commands. 

In [15]:
!docker compsoe --help


Usage:  docker [OPTIONS] COMMAND

A self-sufficient runtime for containers

Common Commands:
  run         Create and run a new container from an image
  exec        Execute a command in a running container
  ps          List containers
  build       Build an image from a Dockerfile
  pull        Download an image from a registry
  push        Upload an image to a registry
  images      List images
  login       Authenticate to a registry
  logout      Log out from a registry
  search      Search Docker Hub for images
  version     Show the Docker version information
  info        Display system-wide information

Management Commands:
  ai*         Ask Gordon - Docker Agent
  builder     Manage builds
  buildx*     Docker Buildx
  compose*    Docker Compose
  container   Manage containers
  context     Manage contexts
  debug*      Get a shell into any image or container
  desktop*    Docker Desktop commands (Beta)
  dev*        Docker Dev Environments
  extension*  Manages Docker exte

To get help for a specific Docker Compose command, we can use `docker compose <COMMAND> --help`, where `<COMMAND>` is the command we need help with.

- For example, `docker compose up --help` displays the help for the Docker Compose `up` command.

In [17]:
!docker compose up --help


Usage:  docker compose up [OPTIONS] [SERVICE...]

Create and start containers

Options:
      --abort-on-container-exit      Stops all containers if any
                                     container was stopped. Incompatible
                                     with -d
      --abort-on-container-failure   Stops all containers if any
                                     container exited with failure.
                                     Incompatible with -d
      --always-recreate-deps         Recreate dependent containers.
                                     Incompatible with --no-recreate.
      --attach stringArray           Restrict attaching to the specified
                                     services. Incompatible with
                                     --attach-dependencies.
      --attach-dependencies          Automatically attach to log output
                                     of dependent services
      --build                        Build images before starting cont

---

## Creating and Starting Resouces in a YAML file with `docker compose up`

To create and start all resources defined in a Docker Compose YAML file, we use the command `docker compose up`.

There are numerous flags that can be used with the `docker compose up` command, but let's look at the most important flags as we create and start the resources in the Docker Compose YAML file we previously examined.

In the cell below we are running the following command:

```bash
docker compose -f ../flixtube/Flixtube.Metadata/docker-compose.yml --project-directory ../flixtube/Flixtube.Metadata up -d --build --force-recreate
```

- `docker compose` is how we invoke the Docker Compose tool.
- `-f <PATH_TO_DOCKER_COMPOSE_YAML_FILE>` is how we specify what Docker Compose YAML file we want to use, where `<PATH_TO_DOCKER_COMPOSE_YAML_FILE>` is the path to the YAML file.
  - In this case we are using the YAML file `../flixtube/Flixtube.Metadata/docker-compose.yml`.
- `--project-directory <PATH_TO_FOLDER_WITH_FILES>` is the path to the folder that contains all the necessary files (Dockerfile, etc.).
  - In this case we are using the folder `../flixtube/Flixtube.Metadata` which contains all the files for the `Flixtube.Metadata` microservice.
- `up` is the Docker Compose command that tells Docker Compose to *bring up* all the resources specified in the YAML file.
  - It will create all services, volumes and networks.
- `-d` tells Docker Compose to start all the services in detached mode.
  - This means all services will run in the backgroud.
    - The current terminal won't be blocked.
    - Any output from the services won't be visible.
  - This is equivalent to using the `-d` flag in the `docker run` command.
- `--build` tells Docker Compose to build the images before starting the containers.
  - If the image for a service has already been built, this flag will rebuild it, ensuring the latest changes to the Dockerfile or context are incorporated.
- `--force_recreate` tells Docker Compose to remove any existing containers for the services and recreate them, even if the configuration or image has not changed.

Now, run the cell below to bring up all resources defined in the Docker Compose YAML file.

In [1]:
!docker compose -f ../flixtube/Flixtube.Metadata/docker-compose.yml --project-directory ../flixtube/Flixtube.Metadata up -d --build --force-recreate

#0 building with "desktop-linux" instance using docker driver

 Service metadata  Building
 Service metadata  Built
 Network flixtubemetadata_default  Creating
 Network flixtubemetadata_default  Created
 Volume "flixtubemetadata_rabbit_data"  Creating
 Volume "flixtubemetadata_rabbit_data"  Created
 Volume "flixtubemetadata_sqlserver_data"  Creating
 Volume "flixtubemetadata_sqlserver_data"  Created
 Container sqlserver  Creating
 Container rabbit  Creating
 Container sqlserver  Created
 Container rabbit  Created
 Container metadata  Creating
 Container metadata  Created
 Container rabbit  Starting
 Container sqlserver  Starting
 Container rabbit  Started
 Container sqlserver  Started
 Container metadata  Starting
 Container metadata  Started




#1 [metadata internal] load build definition from Dockerfile-dev
#1 transferring dockerfile: 454B 0.0s done
#1 DONE 0.0s

#2 [metadata internal] load metadata for mcr.microsoft.com/dotnet/sdk:9.0
#2 DONE 0.5s

#3 [metadata internal] load .dockerignore
#3 transferring context: 393B done
#3 DONE 0.0s

#4 [metadata 1/7] FROM mcr.microsoft.com/dotnet/sdk:9.0@sha256:84fd557bebc64015e731aca1085b92c7619e49bdbe247e57392a43d92276f617
#4 DONE 0.0s

#5 [metadata internal] load build context
#5 transferring context: 4.67kB 0.0s done
#5 DONE 0.0s

#6 [metadata 3/7] COPY ./*.sln ./
#6 CACHED

#7 [metadata 2/7] WORKDIR /src
#7 CACHED

#8 [metadata 4/7] COPY ./Flixtube.Metadata/*.csproj ./Flixtube.Metadata/
#8 CACHED

#9 [metadata 5/7] COPY ./Flixtube.Metadata.UnitTests/*.csproj ./Flixtube.Metadata.UnitTests/
#9 CACHED

#10 [metadata 6/7] COPY ./Flixtube.Metadata.IntegrationTests/*.csproj ./Flixtube.Metadata.IntegrationTests/
#10 CACHED

#11 [metadata 7/7] RUN ["dotnet","restore"]
#11 CACHED

#12 [m

Lets verify that the services (containers) were created and that the `metadata` service logs contain no errors 

In [9]:
!docker ps
!docker logs metadata

CONTAINER ID   IMAGE                                        COMMAND                  CREATED         STATUS         PORTS                                                                                                         NAMES
d3fb1d3a6330   metadata                                     "dotnet watch run --…"   5 minutes ago   Up 5 minutes   0.0.0.0:4030->80/tcp                                                                                          metadata
464687f5416a   rabbitmq:4.0.2-management                    "docker-entrypoint.s…"   5 minutes ago   Up 5 minutes   4369/tcp, 5671/tcp, 0.0.0.0:5672->5672/tcp, 15671/tcp, 15691-15692/tcp, 25672/tcp, 0.0.0.0:15672->15672/tcp   rabbit
3e83449fb8ee   mcr.microsoft.com/mssql/server:2022-latest   "/opt/mssql/bin/perm…"   5 minutes ago   Up 5 minutes   0.0.0.0:1433->1433/tcp                                                                                        sqlserver
dotnet watch ⌚ Polling file watcher is enabled
dotnet watch 🔥 Ho

Let's add a message to the RabbitMQ `uploaded` Exchange

- Visit: http://localhost:15672
- Enter `rabbit_user` as the username.
- Enter `rabbit_password` as the password.
- In the dashboard, click the `Exchanges` tab.
- Then click the `uploaded` exchange under the `Name` column.
- Under `Publish message`, enter the JSON document below into the `Payload` field, and click the `Publish message` button:

```bash
{
    "Id": "6d9e690ad76fe06a3d7ae416",
    "Name": "SampleVideo_2_1280x720_1mb.mp4"
}
```

Let's check if a row was added to the database via the REST API

- Visit: http://localhost:4030/videos
- You should see the following JSON document in your browser:

  ```bash
  [
      {
          "id": "6d9e690ad76fe06a3d7ae416",
          "name": "SampleVideo_2_1280x720_1mb.mp4"
      },
  ]
  ```


---

## Stopping and Destroying Resouces in a YAML file with `docker compose down`

To stop and destroy all resources defined in a Docker Compose YAML file, we use the command `docker compose down`.

There are numerous flags that can be used with the `docker compose down` command, but let's look at the most important flags as we stop and destroy the resources in the Docker Compose YAML file we previously examined.

In the cell below we are running the following command:

```bash
docker compose -f ../flixtube/Flixtube.Metadata/docker-compose.yml --project-directory ../flixtube/Flixtube.Metadata down --rmi local --volumes
```

- `docker compose` is how we invoke the Docker Compose tool.
- `-f <PATH_TO_DOCKER_COMPOSE_YAML_FILE>` is how we specify what Docker Compose YAML file we want to use, where `<PATH_TO_DOCKER_COMPOSE_YAML_FILE>` is the path to the YAML file.
  - In this case we are using the YAML file `../flixtube/Flixtube.Metadata/docker-compose.yml`.
- `--project-directory <PATH_TO_FOLDER_WITH_FILES>` is the path to the folder that contains all the necessary files (Dockerfile, etc.).
  - In this case we are using the folder `../flixtube/Flixtube.Metadata` which contains all the files for the `Flixtube.Metadata` microservice.
- `down` is the Docker Compose command that tells Docker Compose to *tear down* all the resources specified in the YAML file.
  - It will stop all services, and destroy all services volumes and networks.
- `--rmi local` tells Docker Compose to remove **locally-built images** that were created by docker compose during the `docker compose up` process.
  - In this case Docker Compose will remove the `metadata:latest` image.
  - It does not remove images pulled from a remote repository (e.g., Docker Hub).
- `--volumes` tells Docker Compose to remove all named and anonymous volumes declared in the volumes section of the Docker Compose YAML file.
  - This is helpful when you want to ensure that persistent data stored in Docker volumes is deleted. 

Now, run the cell below to tear down all resources defined in the Docker Compose YAML file.

In [10]:
!docker compose -f ../flixtube/Flixtube.Metadata/docker-compose.yml --project-directory ../flixtube/Flixtube.Metadata down --rmi local --volumes

 Container metadata  Stopping
 Container metadata  Stopped
 Container metadata  Removing
 Container metadata  Removed
 Container sqlserver  Stopping
 Container rabbit  Stopping
 Container sqlserver  Stopped
 Container sqlserver  Removing
 Container sqlserver  Removed
 Container rabbit  Stopped
 Container rabbit  Removing
 Container rabbit  Removed
 Volume flixtubemetadata_sqlserver_data  Removing
 Image metadata:latest  Removing
 Volume flixtubemetadata_rabbit_data  Removing
 Network flixtubemetadata_default  Removing
 Image metadata:latest  Removed
 Volume flixtubemetadata_sqlserver_data  Removed
 Volume flixtubemetadata_rabbit_data  Removed
 Network flixtubemetadata_default  Removed


Let's verify that the services (containers) where stopped and destroyed, and that the `metadata:latest` image was deleted

In [ ]:
!docker ps -a
!docker images

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES
REPOSITORY   TAG       IMAGE ID   CREATED   SIZE


## Docker Compose Documentation

In this notebook, we have covered the most important Dockerfile Compose concepts.

To learn more about [Docker Compose](https://docs.docker.com/compose):

- Visit [https://docs.docker.com/compose/intro/compose-application-model](https://docs.docker.com/compose/intro/compose-application-model) to find out how Docker Compose works.
- Visit [https://docs.docker.com/compose/gettingstarted](https://docs.docker.com/compose/gettingstarted) for the Docker Compose quick start.

- Visit [https://docs.docker.com/reference/compose-file](https://docs.docker.com/reference/compose-file) for the Docker Compose YAML File reference.

- Visit [https://docs.docker.com/reference/cli/docker/compose](https://docs.docker.com/reference/cli/docker/compose) for the Docker Compose CLI reference.